In [0]:
%run ../../config/utils

In [0]:
import sys
sys.path.append("..")
sys.path.append('../..')
import datetime

import os
import yaml
import lib.misc as misc

In [0]:
dbutils.widgets.text("in_home_date", 'automatic', "In Home Date") # create the widget if missing

In [0]:

config_path = "config/config.yml"
if not os.path.exists(config_path):
    raise FileNotFoundError(f"Missing configuration: {config_path}") 

with open(config_path) as config_file:
    config = yaml.load(config_file, Loader=yaml.FullLoader)


In [0]:
# potential remove:

# cube_path = "Code_and_Data_repo/CUBES/customer_cube_2019-12-02/customer_cube"
# config["shared"]["cube_path"] = cube_path

current_date = datetime.datetime.now()
days_to_subtract = (current_date.weekday() - 6) % 7
curr_dt = (current_date - datetime.timedelta(days=days_to_subtract)).strftime("%Y-%m-%d")

config["shared"]["run_name"] = curr_dt

In [0]:
def get_max_file_date(table_name, column, format="%Y-%m-%d"):
    last_date = spark.table(table_name).agg({column: "max"}).collect()[0][0]
    if last_date:
        return last_date.strftime(format) if hasattr(last_date, "strftime") else str(last_date)
    return last_date

def get_next_file_date(table_name, column, date=None, format="%Y-%m-%d"):
    df = spark.table(table_name)
    next_row = (
        df.filter(f"{column} > '{date}'")
        .orderBy(column)
        .select(column)
        .limit(1)
        .collect()
    )
    if next_row:
        val = next_row[0][0]
        return val.strftime(format) if hasattr(val, "strftime") else str(val)
    return None

In [0]:
# Associated Campaign In-Home Date (YYYY-mm-dd), used to determine the seasonality of the model.
# If left to the default of 'automatic', uses a date 6 weeks ahead of the latest cube.

in_home_date = dbutils.widgets.get("in_home_date")


#-----------------------------------------------------------------------------------------------------
if in_home_date == "automatic":

    last_cube_date_str = get_max_file_date(customer_cube_archive, 'run_date')

else:

    last_date = (
        datetime.datetime.strptime(in_home_date, "%Y-%m-%d")
        - datetime.timedelta(6 * 7)
    ).strftime("%Y-%m-%d")

    last_cube_date_str = get_next_file_date(customer_cube_archive, 'run_date', last_date   )
    if not last_cube_date_str:
        print(
            "No data file within six weeks of the In-Home date. \n \
            Using the latest data file instead."
        )
        last_cube_date_str = get_max_file_date(customer_cube_archive, 'run_date')

In [0]:
(
    config["etl"]["start_date"],
    config["etl"]["end_date"],
) = misc.get_previous_fiscal_weekend(13 * 365 / 12, last_cube_date_str)

# These were originally set to be configured only when the env is not dev but we need to run processes as prod to evaluate performance so we removed the filter
config["predict"]["weeks_to_predict"]   = [config["etl"]["end_date"]]
config["predict"]["trip_model_date"]    = datetime.date.today().strftime("%Y%m%d")
config["predict"]["spend_model_date"]   = datetime.date.today().strftime("%Y%m%d")

In [0]:
with open(config_path, "w") as config_file:
    yaml.dump(config, config_file)